# SentimentFlow — Notebook 02: Agregaciones para Dashboard

Lee todos los Parquet procesados y calcula las métricas que el dashboard Streamlit consulta en Azure SQL.

Se ejecuta de forma programada (cron diario) o manualmente.

**Responsable:** Persona 2

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

STORAGE_ACCOUNT  = dbutils.secrets.get(scope='sentimentflow', key='storage_account_name')
STORAGE_KEY       = dbutils.secrets.get(scope='sentimentflow', key='storage_account_key')
SQL_JDBC_URL      = dbutils.secrets.get(scope='sentimentflow', key='sql_jdbc_url')
SQL_USER          = dbutils.secrets.get(scope='sentimentflow', key='sql_user')
SQL_PASSWORD      = dbutils.secrets.get(scope='sentimentflow', key='sql_password')

spark.conf.set(
    f'fs.azure.account.key.{STORAGE_ACCOUNT}.blob.core.windows.net',
    STORAGE_KEY
)
PROCESSED_PATH = f'wasbs://processed-reviews@{STORAGE_ACCOUNT}.blob.core.windows.net/'

In [ ]:
# ─── Leer todos los Parquet procesados ────────────────────────────────────────
df = spark.read.parquet(PROCESSED_PATH)
print(f'Total registros acumulados: {df.count():,}')
df.printSchema()

In [ ]:
# ─── Tabla 1: métricas por producto ───────────────────────────────────────────
df_by_product = df.groupBy('product_id', 'product_name', 'category').agg(
    F.count('*').alias('total_reviews'),
    F.round(F.avg('rating'), 2).alias('avg_rating'),
    F.round(F.avg('sentiment_score'), 4).alias('avg_sentiment'),
    F.sum(F.when(F.col('sentiment_label') == 'Positivo', 1).otherwise(0)).alias('positivos'),
    F.sum(F.when(F.col('sentiment_label') == 'Neutro',   1).otherwise(0)).alias('neutros'),
    F.sum(F.when(F.col('sentiment_label') == 'Negativo', 1).otherwise(0)).alias('negativos'),
    F.max('processed_at').alias('last_updated'),
)

df_by_product.show(10)

In [ ]:
# ─── Tabla 2: evolución temporal (por hora) ───────────────────────────────────
df_timeseries = df.withColumn(
    'hour_bucket', F.date_trunc('hour', F.col('event_ts'))
).groupBy('hour_bucket', 'category').agg(
    F.count('*').alias('num_reviews'),
    F.round(F.avg('sentiment_score'), 4).alias('avg_sentiment'),
    F.round(F.avg('rating'), 2).alias('avg_rating'),
).orderBy('hour_bucket')

df_timeseries.show(10)

In [ ]:
# ─── Tabla 3: sentimiento por país ────────────────────────────────────────────
df_by_country = df.groupBy('country').agg(
    F.count('*').alias('total_reviews'),
    F.round(F.avg('sentiment_score'), 4).alias('avg_sentiment'),
).orderBy('total_reviews', ascending=False)

df_by_country.show(15)

In [ ]:
# ─── Escribir agregaciones en Azure SQL (JDBC) ────────────────────────────────
jdbc_props = {
    'user': SQL_USER,
    'password': SQL_PASSWORD,
    'driver': 'com.microsoft.sqlserver.jdbc.SQLServerDriver',
}

for df_agg, table_name in [
    (df_by_product,  'dbo.agg_by_product'),
    (df_timeseries,  'dbo.agg_timeseries'),
    (df_by_country,  'dbo.agg_by_country'),
]:
    (
        df_agg.write
        .mode('overwrite')
        .jdbc(url=SQL_JDBC_URL, table=table_name, properties=jdbc_props)
    )
    print(f'✓ Tabla {table_name} actualizada ({df_agg.count()} filas)')

print('\nAgregaciones completadas.')
dbutils.notebook.exit('OK')